In [ ]:
# =============================================================================
# XGBoost Ignition + XGBoost Binary Cause Models — Ontario
# Train: 2010–2020  |  Validation: 2021–2022
# Ignition: XGBoost (model_ignition_final.pkl, CV threshold)
# Cause: two binary XGBoost models (Human vs Other, Natural vs Other)
#        combined via three-way decision rule
# =============================================================================

import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import (classification_report, roc_auc_score,
                             confusion_matrix, balanced_accuracy_score,
                             precision_score, recall_score, f1_score,
                             average_precision_score)
from sklearn.inspection import permutation_importance
from sklearn.utils.class_weight import compute_sample_weight
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import glob
import os
import joblib
import time
import warnings
warnings.filterwarnings('ignore')

# ── Configuration ─────────────────────────────────────────────────────────────
DATA_DIR = r'C:\Users\lambe\RU_Thesis_2026\ml_data'
os.makedirs('models',  exist_ok=True)
os.makedirs('figures', exist_ok=True)

DYNAMIC_FEATURES = [
    'temp_max', 'rh_min', 'vpd_mean', 'wind_speed_mean',
    'precip_sum', 'solar_rad_mj', 'rh_7d', 'temp_7d',
    'precip_30d', 'recovery_value'
]
STATIC_FEATURES = [
    'elevation_m', 'slope_deg', 'land_cover',
    'pop_density', 'road_proximity_m'
]
FEATURES = DYNAMIC_FEATURES + STATIC_FEATURES

FEATURE_LABELS = {
    'temp_max'        : 'Daily max temperature (°C)',
    'rh_min'          : 'Daily min relative humidity (%)',
    'vpd_mean'        : 'Mean VPD (kPa)',
    'wind_speed_mean' : 'Mean wind speed (m/s)',
    'precip_sum'      : 'Daily precipitation (mm)',
    'solar_rad_mj'    : 'Solar radiation (MJ/m²)',
    'rh_7d'           : '7-day mean RH (%)',
    'temp_7d'         : '7-day mean temperature (°C)',
    'precip_30d'      : '30-day precipitation (mm)',
    'recovery_value'  : 'Post-fire recovery [0–1]',
    'elevation_m'     : 'Elevation (m)',
    'slope_deg'       : 'Slope (degrees)',
    'land_cover'      : 'Land cover class',
    'pop_density'     : 'Population density',
    'road_proximity_m': 'Distance to road (m)',
}

DOY_START_HARD      = 91
DOY_END_HARD        = 310
TRAIN_YEARS         = list(range(2010, 2021))
VAL_YEARS           = [2021, 2022]
UNCERTAIN_THRESHOLD = 0.35
RANDOM_STATE        = 42

NEG_NAMES = {
    'human'  : 'Other (Natural/Undetermined)',
    'natural': 'Other (Human/Undetermined)',
}

COL_DYNAMIC  = '#27ae60'
COL_STATIC   = '#e67e22'
COL_HUMAN    = '#e74c3c'
COL_NATURAL  = '#2980b9'
COL_GRID     = '#eeeeee'
FONT_TITLE   = 12
FONT_LABEL   = 10
FONT_TICK    = 9

SOURCE_IGN   = ('Ontario wildfire ignition model  |  XGBoost (SMOTE + Optuna)  |  '
                'Training: 2010–2020  |  Validation: 2021–2022')
SOURCE_CAUSE = ('Ontario wildfire cause model  |  XGBoost binary  |  '
                'XGBoost ignition base  |  Validation: 2021–2022')

def feature_colour(feat):
    return COL_DYNAMIC if feat in DYNAMIC_FEATURES else COL_STATIC

# =============================================================================
# STEP 1: Load XGBoost ignition model
# =============================================================================
print("="*60)
print("STEP 1: Loading XGBoost ignition model")
print("="*60)

bundle       = joblib.load('models/model_ignition_final.pkl')
model_xgb    = bundle['model']
CV_THRESHOLD = bundle['cv_threshold']

print(f"  Model     : model_ignition_final.pkl")
print(f"  Threshold : {CV_THRESHOLD:.4f}")
print(f"  Optuna AUC: {bundle.get('optuna_auc', 'N/A')}")
print(f"  SMOTE ratio: {bundle.get('smote_ratio', 'N/A')}")

# =============================================================================
# STEP 2: Load data
# =============================================================================
print("\n" + "="*60)
print("STEP 2: Loading data")
print("="*60)

def load_years(years, split_label):
    all_dfs = []
    for year in years:
        pattern  = os.path.join(DATA_DIR, f'ML_{split_label}_{year}??.csv')
        files    = sorted(glob.glob(pattern))
        year_dfs = []
        for f in files:
            try:
                year_dfs.append(pd.read_csv(f))
            except Exception as e:
                print(f"  WARNING: {os.path.basename(f)}: {e}")
        if year_dfs:
            df_year = pd.concat(year_dfs, ignore_index=True)
            n_fire  = (df_year['target_ignition'] == 1).sum()
            print(f"  {year}: {len(df_year):>10,} rows  ({n_fire:,} fires)")
            all_dfs.append(df_year)
    return pd.concat(all_dfs, ignore_index=True)

def clean_and_filter(df, name):
    required = FEATURES + ['target_ignition', 'fire_cause', 'date']
    df = df.dropna(subset=required)
    df['target_ignition'] = df['target_ignition'].astype(int)
    df['fire_cause']      = df['fire_cause'].astype(int)
    df['land_cover']      = df['land_cover'].astype(int)
    df = df[df['target_ignition'].isin([0, 1])]
    df = df[df['fire_cause'].between(0, 5)]
    df['date_str'] = df['date'].astype(int).astype(str)
    df['doy']      = pd.to_datetime(
        df['date_str'], format='%Y%m%d', errors='coerce').dt.dayofyear
    n_before = len(df)
    df = df[(df['doy'] >= DOY_START_HARD) &
            (df['doy'] <= DOY_END_HARD)].copy()
    df = df.drop(columns=['date_str', 'doy'])
    df = df.sort_values('date').reset_index(drop=True)
    print(f"  {name}: {n_before:,} → {len(df):,} rows after season filter")
    return df

print("\nLoading training data (2010–2020)...")
df_train = clean_and_filter(load_years(TRAIN_YEARS, 'train'), 'train')

print("\nLoading validation data (2021–2022)...")
df_val = clean_and_filter(load_years(VAL_YEARS, 'val'), 'val')

X_train     = df_train[FEATURES].values
y_ign_train = df_train['target_ignition'].values
X_val       = df_val[FEATURES].values
y_ign_val   = df_val['target_ignition'].values

fire_tr_mask = df_train['target_ignition'] == 1
fire_vl_mask = df_val['target_ignition']   == 1

print(f"\n  Train: {X_train.shape}  fires: {fire_tr_mask.sum():,}")
print(f"  Val  : {X_val.shape}    fires: {fire_vl_mask.sum():,}")

# =============================================================================
# STEP 3: Run XGBoost ignition predictions
# =============================================================================
print("\n" + "="*60)
print("STEP 3: XGBoost ignition predictions")
print(f"  CV threshold: {CV_THRESHOLD:.4f}")
print("="*60)

y_prob_ign = model_xgb.predict_proba(X_val)[:, 1]
y_pred_ign = (y_prob_ign >= CV_THRESHOLD).astype(int)

tn_i, fp_i, fn_i, tp_i = confusion_matrix(y_ign_val, y_pred_ign).ravel()

print(f"\n  Ignition validation results:")
print(f"    ROC-AUC  : {roc_auc_score(y_ign_val, y_prob_ign):.4f}")
print(f"    Recall   : {tp_i/(tp_i+fn_i):.4f}  "
      f"({tp_i} of {tp_i+fn_i} fires detected)")
print(f"    Precision: {tp_i/(tp_i+fp_i) if (tp_i+fp_i)>0 else 0:.4f}")
print(f"    FPR      : {fp_i/(fp_i+tn_i):.4f}")
print(f"    TN={tn_i:,}  FP={fp_i:,}  FN={fn_i:,}  TP={tp_i:,}")
print(classification_report(y_ign_val, y_pred_ign,
                            target_names=['No fire', 'Fire'],
                            digits=4, zero_division=0))

# =============================================================================
# STEP 4: Identify predicted fire pixels for cause modelling
# =============================================================================
print("\n" + "="*60)
print("STEP 4: Identifying predicted fire pixels for cause modelling")
print("="*60)

# Training cause: actual confirmed fire pixels with known cause
X_cause_train     = df_train.loc[fire_tr_mask, FEATURES].values
y_cause_train     = df_train.loc[fire_tr_mask, 'fire_cause'].values

# Validation cause: XGBoost-predicted fire pixels
mask_pred_fire    = y_pred_ign == 1
X_pred_fire_val   = X_val[mask_pred_fire]
y_cause_pred_true = df_val.loc[mask_pred_fire, 'fire_cause'].values
y_ign_pred_true   = df_val.loc[mask_pred_fire, 'target_ignition'].values

# Actual fires within the predicted set (exclude false positives)
actual_mask   = y_ign_pred_true == 1
X_cause_val_a = X_pred_fire_val[actual_mask]
y_cause_val_a = y_cause_pred_true[actual_mask]

print(f"  Cause train pixels (actual fires) : {len(X_cause_train):,}")
print(f"  XGBoost predicted fire pixels     : {mask_pred_fire.sum():,}")
print(f"    Actual fires (TP)               : "
      f"{(y_ign_pred_true==1).sum():,}")
print(f"    False positives                 : "
      f"{(y_ign_pred_true==0).sum():,}")
print(f"  Cause val set (actual fires only) : {len(X_cause_val_a):,}")

print(f"\n  Cause distribution in training set:")
for code, label in {1:'Undetermined', 2:'Human',
                    3:'Natural', 4:'H-PB'}.items():
    n = (y_cause_train == code).sum()
    if n > 0:
        print(f"    Code {code} ({label:<15}): {n:,}  "
              f"({n/len(y_cause_train)*100:.1f}%)")

# =============================================================================
# STEP 5: Train binary XGBoost cause models
#         WITH Optuna hyperparameter tuning + CV threshold optimization
# =============================================================================
print("\n" + "="*60)
print("STEP 5: Training binary XGBoost cause models")
print("  Optuna hyperparameter tuning + CV threshold optimization")
print("="*60)

from sklearn.model_selection import StratifiedKFold
import optuna

# ── Cause model configuration ─────────────────────────────────────────────────
N_TRIALS_CAUSE   = 50
N_FOLDS_OPTUNA   = 3
N_FOLDS_THRESH   = 5
MIN_RECALL_CAUSE = 0.60    # lower floor than ignition — cause is harder
                            # adjust to 0.50 if either model cannot achieve 0.60
N_THR_CANDIDATES = 200
RANDOM_STATE      = 42

optuna.logging.set_verbosity(optuna.logging.WARNING)

cause_models = {}

for model_key, pos_label, pos_name, neg_name, color in [
    ('human',   2, 'Human',
     NEG_NAMES['human'],   COL_HUMAN),
    ('natural', 3, 'Natural (lightning)',
     NEG_NAMES['natural'], COL_NATURAL),
]:
    print(f"\n{'='*55}")
    print(f"  Binary XGBoost: {pos_name} vs Other")
    print(f"{'='*55}")

    # Binary labels
    y_tr_bin = (y_cause_train == pos_label).astype(int)
    y_vl_bin = (y_cause_val_a == pos_label).astype(int)

    n_pos_tr = y_tr_bin.sum()
    n_neg_tr = (y_tr_bin == 0).sum()
    ratio    = n_neg_tr / max(n_pos_tr, 1)

    print(f"  Train — pos: {n_pos_tr:,}  neg: {n_neg_tr:,}  "
          f"ratio: {ratio:.1f}:1")
    print(f"  Val actual fires: {len(X_cause_val_a):,}  "
          f"pos: {y_vl_bin.sum():,}")

    # ── Optuna objective ───────────────────────────────────────────────────────
    # Optimises ROC-AUC on the unaugmented validation fold
    # (cause dataset is small enough that full CV is fast)
    def make_cause_objective(y_tr, X_tr, X_vl, y_vl, spw):
        def objective(trial):
            params = {
                'objective'            : 'binary:logistic',
                'eval_metric'          : 'auc',
                'scale_pos_weight'     : spw,
                'n_estimators'         : trial.suggest_int(
                    'n_estimators', 100, 600),
                'max_depth'            : trial.suggest_int(
                    'max_depth', 3, 7),
                'learning_rate'        : trial.suggest_float(
                    'learning_rate', 0.01, 0.3, log=True),
                'subsample'            : trial.suggest_float(
                    'subsample', 0.5, 1.0),
                'colsample_bytree'     : trial.suggest_float(
                    'colsample_bytree', 0.5, 1.0),
                'min_child_weight'     : trial.suggest_int(
                    'min_child_weight', 1, 20),
                'gamma'                : trial.suggest_float(
                    'gamma', 0.0, 5.0),
                'reg_alpha'            : trial.suggest_float(
                    'reg_alpha', 1e-4, 10.0, log=True),
                'reg_lambda'           : trial.suggest_float(
                    'reg_lambda', 1e-4, 10.0, log=True),
                'random_state'         : RANDOM_STATE,
                'n_jobs'               : -1,
                'verbosity'            : 0,
                'early_stopping_rounds': 15,
            }

            skf  = StratifiedKFold(
                n_splits=N_FOLDS_OPTUNA,
                shuffle=True, random_state=RANDOM_STATE)
            aucs = []

            for tr_idx, vl_idx in skf.split(X_tr, y_tr):
                n_est = params.pop('n_estimators')
                early = params.pop('early_stopping_rounds')
                m = xgb.XGBClassifier(
                    n_estimators=n_est,
                    early_stopping_rounds=early,
                    **params)
                m.fit(X_tr[tr_idx], y_tr[tr_idx],
                      eval_set=[(X_tr[vl_idx], y_tr[vl_idx])],
                      verbose=False)
                aucs.append(roc_auc_score(
                    y_tr[vl_idx],
                    m.predict_proba(X_tr[vl_idx])[:, 1]))
                params['n_estimators']          = n_est
                params['early_stopping_rounds'] = early

            return np.mean(aucs)
        return objective

    print(f"\n  Running Optuna ({N_TRIALS_CAUSE} trials, "
          f"{N_FOLDS_OPTUNA}-fold CV)...")
    t0    = time.time()
    study = optuna.create_study(
        direction  = 'maximize',
        study_name = f'xgb_cause_{model_key}',
        sampler    = optuna.samplers.TPESampler(seed=RANDOM_STATE))
    study.optimize(
        make_cause_objective(
            y_tr_bin, X_cause_train,
            X_cause_val_a, y_vl_bin,
            ratio),
        n_trials          = N_TRIALS_CAUSE,
        timeout           = 1800,
        show_progress_bar = True)

    best_params = study.best_params
    best_auc_cv = study.best_value
    print(f"  Optuna done in {(time.time()-t0)/60:.1f} min  "
          f"best CV AUC: {best_auc_cv:.4f}")
    print(f"  Best parameters:")
    for k, v in best_params.items():
        if isinstance(v, float):
            print(f"    {k:<25} {v:.6f}")
        else:
            print(f"    {k:<25} {v}")

    joblib.dump(study,
                f'models/optuna_study_cause_{model_key}.pkl')

    # ── Retrain on full cause training set with best params ────────────────────
    print(f"\n  Retraining final model on full cause training set...")
    sample_weights = compute_sample_weight('balanced', y_tr_bin)

    t0 = time.time()
    model = xgb.XGBClassifier(
        objective             = 'binary:logistic',
        eval_metric           = 'auc',
        scale_pos_weight      = ratio,
        early_stopping_rounds = 20,
        random_state          = RANDOM_STATE,
        n_jobs                = -1,
        verbosity             = 1,
        **best_params
    )
    model.fit(
        X_cause_train, y_tr_bin,
        sample_weight = sample_weights,
        eval_set      = [(X_cause_val_a, y_vl_bin)],
        verbose       = 50
    )
    print(f"  Training time : {(time.time()-t0)/60:.1f} min")
    print(f"  Best iteration: {model.best_iteration}")

    # ── CV threshold optimization ──────────────────────────────────────────────
    # Finds threshold that maximises precision at recall >= MIN_RECALL_CAUSE
    # Uses the cause training set (not augmented) for threshold CV
    print(f"\n  CV threshold optimization "
          f"(min recall = {MIN_RECALL_CAUSE})...")

    skf_t      = StratifiedKFold(
        n_splits=N_FOLDS_THRESH,
        shuffle=True, random_state=RANDOM_STATE)
    candidates = np.linspace(0.01, 0.99, N_THR_CANDIDATES)
    fold_thrs  = []

    for fold_idx, (tr_idx, vl_idx) in enumerate(
            skf_t.split(X_cause_train, y_tr_bin)):
        y_prob_f  = model.predict_proba(X_cause_train[vl_idx])[:, 1]
        best_thr, best_prec = 0.5, 0.0
        for thr in candidates:
            y_t  = (y_prob_f >= thr).astype(int)
            rec  = recall_score(
                y_tr_bin[vl_idx], y_t, zero_division=0)
            prec = precision_score(
                y_tr_bin[vl_idx], y_t, zero_division=0)
            if rec >= MIN_RECALL_CAUSE and prec > best_prec:
                best_prec, best_thr = prec, thr
        fold_thrs.append(best_thr)
        print(f"    Fold {fold_idx+1}: "
              f"threshold={best_thr:.3f}  precision={best_prec:.4f}")

    cv_thr     = float(np.mean(fold_thrs))
    cv_thr_std = float(np.std(fold_thrs))
    print(f"  CV threshold: {cv_thr:.4f} ± {cv_thr_std:.4f}")

    # ── Evaluate at default 0.5 and CV threshold ───────────────────────────────
    y_prob_vl = model.predict_proba(X_cause_val_a)[:, 1]
    auc_vl    = roc_auc_score(y_vl_bin, y_prob_vl)
    ap_vl     = average_precision_score(y_vl_bin, y_prob_vl)

    for label_thr, thr in [('Default (0.50)', 0.50),
                             (f'CV ({cv_thr:.3f})', cv_thr)]:
        y_pred_thr = (y_prob_vl >= thr).astype(int)
        print(f"\n  Evaluation at {label_thr}:")
        print(classification_report(
            y_vl_bin, y_pred_thr,
            target_names=[neg_name, pos_name],
            digits=4, zero_division=0))

    # ── Save ───────────────────────────────────────────────────────────────────
    save_path = f'models/model_xgb_cause_binary_{model_key}.pkl'
    joblib.dump({
        'model'             : model,
        'pos_label'         : pos_label,
        'pos_name'          : pos_name,
        'neg_name'          : neg_name,
        'features'          : FEATURES,
        'cv_threshold'      : cv_thr,
        'cv_threshold_std'  : cv_thr_std,
        'val_auc'           : auc_vl,
        'val_ap'            : ap_vl,
        'best_params'       : best_params,
        'optuna_auc'        : best_auc_cv,
        'min_recall_cause'  : MIN_RECALL_CAUSE,
        'ignition_model'    : 'model_ignition_final.pkl',
        'ignition_thr'      : CV_THRESHOLD,
    }, save_path)
    print(f"\n  Saved: {save_path}")

    cause_models[model_key] = {
        'model'    : model,
        'pos_label': pos_label,
        'pos_name' : pos_name,
        'neg_name' : neg_name,
        'color'    : color,
        'val_auc'  : auc_vl,
        'cv_thr'   : cv_thr,
        'cv_thr_std': cv_thr_std,
    }

# =============================================================================
# STEP 5b: Detailed binary cause evaluation
# =============================================================================
print("\n" + "="*60)
print("STEP 5b: Detailed binary cause model evaluation")
print("="*60)

binary_eval = {}

for model_key, info in cause_models.items():
    pos_label = info['pos_label']
    pos_name  = info['pos_name']
    neg_name  = info.get('neg_name', NEG_NAMES.get(model_key, 'Other'))

    y_bin_true = (y_cause_val_a == pos_label).astype(int)
    y_bin_prob = info['model'].predict_proba(X_cause_val_a)[:, 1]
    y_bin_pred = (y_bin_prob >= info['cv_thr']).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_bin_true, y_bin_pred).ravel()
    bal_acc = balanced_accuracy_score(y_bin_true, y_bin_pred)
    prec    = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec     = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1      = (2 * prec * rec / (prec + rec)
               if (prec + rec) > 0 else 0.0)
    fpr_bin = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    spec    = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    auc_val = roc_auc_score(y_bin_true, y_bin_prob)

    binary_eval[model_key] = {
        'pos_name': pos_name,
        'neg_name': neg_name,
        'bal_acc' : bal_acc,
        'val_auc' : auc_val,
        'prec'    : prec,
        'rec'     : rec,
        'spec'    : spec,
        'f1'      : f1,
        'fpr'     : fpr_bin,
        'tp': tp, 'fp': fp, 'fn': fn, 'tn': tn,
        'n_total' : len(y_bin_true),
        'n_pos'   : int(y_bin_true.sum()),
    }

    print(f"\n{'─'*55}")
    print(f"  {pos_name} vs Other — Confusion Matrix")
    print(f"  Evaluated on actual fires in XGBoost predicted set "
          f"(n={len(y_bin_true):,})")
    print(f"  Positive class: {pos_name}  "
          f"(n={y_bin_true.sum():,},  "
          f"{y_bin_true.mean()*100:.1f}%)")
    print(f"{'─'*55}")
    print(f"\n  Confusion matrix (threshold = 0.50):")
    print(f"  {'':25} {'Pred Other':>14} {'Pred '+pos_name[:7]:>14}")
    print(f"  {'─'*55}")
    print(f"  {'Actual Other':<25} {tn:>14,} {fp:>14,}")
    print(f"  {'Actual '+pos_name:<25} {fn:>14,} {tp:>14,}")
    print(f"  {'─'*55}")
    print(f"\n  Performance metrics:")
    print(f"    Balanced accuracy : {bal_acc:.4f}")
    print(f"    ROC-AUC           : {auc_val:.4f}")
    print(f"    Precision         : {prec:.4f}")
    print(f"    Recall            : {rec:.4f}")
    print(f"    Specificity       : {spec:.4f}")
    print(f"    F1 score          : {f1:.4f}")
    print(f"    FPR               : {fpr_bin:.4f}")
    print(f"    TP={tp:,}  FP={fp:,}  FN={fn:,}  TN={tn:,}")

# ── Side-by-side comparison ────────────────────────────────────────────────────
print(f"\n{'='*65}")
print("BINARY CAUSE MODEL COMPARISON — SIDE BY SIDE")
print(f"{'='*65}")
print(f"  {'Metric':<22} {'Human vs Other':>16} {'Natural vs Other':>18}")
print(f"  {'─'*58}")
for label, key in [
    ('Balanced accuracy', 'bal_acc'),
    ('ROC-AUC',           'val_auc'),
    ('Precision',         'prec'),
    ('Recall',            'rec'),
    ('Specificity',       'spec'),
    ('F1 score',          'f1'),
    ('FPR',               'fpr'),
]:
    print(f"  {label:<22} "
          f"{binary_eval['human'][key]:>16.4f} "
          f"{binary_eval['natural'][key]:>18.4f}")
print(f"  {'─'*58}")
for label, key in [('TP','tp'),('FP','fp'),('FN','fn'),('TN','tn')]:
    print(f"  {label:<22} "
          f"{binary_eval['human'][key]:>16,} "
          f"{binary_eval['natural'][key]:>18,}")

# =============================================================================
# STEP 6: Three-way decision rule
# =============================================================================
print("\n" + "="*60)
print("STEP 6: Three-way cause prediction")
print(f"  Uncertainty threshold: {UNCERTAIN_THRESHOLD}")
print("="*60)

p_human   = cause_models['human']['model'].predict_proba(
    X_pred_fire_val)[:, 1]
p_natural = cause_models['natural']['model'].predict_proba(
    X_pred_fire_val)[:, 1]

THR_HUMAN   = cause_models['human']['cv_thr']
THR_NATURAL = cause_models['natural']['cv_thr']

def predict_three_way_cv(p_hum, p_nat, thr_hum, thr_nat):
    """
    Uses per-model CV thresholds rather than a single fixed threshold.
    Natural wins if P(Natural) > P(Human) AND P(Natural) >= thr_nat.
    Human wins   if P(Human) >= P(Natural) AND P(Human)   >= thr_hum.
    Otherwise Uncertain.
    """
    pred = np.zeros(len(p_hum), dtype=int)
    pred[(p_nat > p_hum)  & (p_nat >= thr_nat)] = 3   # Natural
    pred[(p_hum >= p_nat) & (p_hum >= thr_hum)] = 2   # Human
    return pred

y_pred_cause = predict_three_way_cv(
    p_human, p_natural, THR_HUMAN, THR_NATURAL)

print(f"\n  Per-model CV thresholds:")
print(f"    Human   : {THR_HUMAN:.4f} ± "
      f"{cause_models['human']['cv_thr_std']:.4f}")
print(f"    Natural : {THR_NATURAL:.4f} ± "
      f"{cause_models['natural']['cv_thr_std']:.4f}")

# Evaluate on actual fires only
y_true_eval = y_cause_pred_true[actual_mask]
y_pred_eval = y_pred_cause[actual_mask]

target_names_map = {0: 'Uncertain', 2: 'Human', 3: 'Natural'}
present_classes  = sorted(set(
    list(y_true_eval[np.isin(y_true_eval, [2, 3])]) +
    list(y_pred_eval)))
present_classes  = [c for c in present_classes if c in target_names_map]

print(f"\n  Prediction distribution (all XGBoost predicted fire pixels):")
for code, name in [(2,'Human'), (3,'Natural'), (0,'Uncertain')]:
    n = (y_pred_cause == code).sum()
    print(f"    {name:<12}: {n:,}  ({n/len(y_pred_cause)*100:.1f}%)")

print(f"\n  Classification report (actual fires in predicted set):")
print(classification_report(
    y_true_eval, y_pred_eval,
    labels       = present_classes,
    target_names = [target_names_map[c] for c in present_classes],
    digits=4, zero_division=0))

# =============================================================================
# STEP 6b: Three-way accuracy
# =============================================================================
print(f"\n{'='*65}")
print("STEP 6b: Three-way cause prediction accuracy")
print(f"  Uncertainty threshold : {UNCERTAIN_THRESHOLD}")
print(f"  Evaluated on          : actual fires in XGBoost predicted set "
      f"(n={len(y_true_eval):,})")
print(f"{'='*65}")

n_total         = len(y_true_eval)
n_uncertain     = (y_pred_eval == 0).sum()
n_known         = (y_pred_eval != 0).sum()
correct_mask    = y_pred_eval == y_true_eval
n_correct       = correct_mask.sum()
n_correct_known = (correct_mask & (y_pred_eval != 0)).sum()
overall_acc     = n_correct / n_total  if n_total > 0 else 0.0
known_acc       = n_correct_known / n_known if n_known > 0 else 0.0

hn_mask      = np.isin(y_true_eval, [2, 3])
bal_acc_3way = balanced_accuracy_score(
    y_true_eval[hn_mask], y_pred_eval[hn_mask])

print(f"\n  Overall accuracy (Uncertain counted as wrong):")
print(f"    {n_correct:,} / {n_total:,}  =  {overall_acc:.4f}  "
      f"({overall_acc*100:.2f}%)")
print(f"\n  Accuracy on classified predictions only (excl. Uncertain):")
print(f"    Classified : {n_known:,} / {n_total:,}  "
      f"({n_known/n_total*100:.1f}% of fires received a label)")
print(f"    Correct    : {n_correct_known:,} / {n_known:,}  "
      f"=  {known_acc:.4f}  ({known_acc*100:.2f}%)")
print(f"\n  Balanced accuracy (Human + Natural labels only):")
print(f"    {bal_acc_3way:.4f}")

print(f"\n  Per-class breakdown:")
print(f"  {'True class':<16} {'N actual':>10} {'Pred Human':>12} "
      f"{'Pred Natural':>14} {'Pred Uncertain':>16} {'Accuracy':>10}")
print(f"  {'─'*80}")
for true_code, true_name in [(2,'Human'), (3,'Natural')]:
    mask_true  = y_true_eval == true_code
    n_true     = mask_true.sum()
    if n_true == 0:
        continue
    n_pred_hum = ((y_pred_eval == 2) & mask_true).sum()
    n_pred_nat = ((y_pred_eval == 3) & mask_true).sum()
    n_pred_unc = ((y_pred_eval == 0) & mask_true).sum()
    acc_c      = ((y_pred_eval == true_code) & mask_true).sum() / n_true
    print(f"  {true_name:<16} {n_true:>10,} {n_pred_hum:>12,} "
          f"{n_pred_nat:>14,} {n_pred_unc:>16,} {acc_c:>10.4f}")
print(f"  {'─'*80}")

# =============================================================================
# STEP 7: Feature importance figures
# =============================================================================
print("\n" + "="*60)
print("STEP 7: Generating feature importance figures")
print("="*60)

def plot_xgb_native_importance(ax, model, title):
    """XGBoost native gain importance — no error bars (single model)."""
    imps   = model.feature_importances_
    idx    = np.argsort(imps)
    labs   = [FEATURE_LABELS[FEATURES[i]] for i in idx]
    colors = [feature_colour(FEATURES[i]) for i in idx]
    vals   = imps[idx]
    ax.barh(range(len(vals)), vals,
            color=colors, edgecolor='white', linewidth=0.5,
            height=0.7, zorder=3)
    for i, v in enumerate(vals):
        ax.text(v + vals.max()*0.01, i, f'{v:.3f}',
                va='center', ha='left', fontsize=7.5, color='#333333')
    ax.set_yticks(range(len(labs)))
    ax.set_yticklabels(labs, fontsize=FONT_TICK)
    ax.set_xlabel('Importance (gain)', fontsize=FONT_LABEL)
    ax.set_title(title, fontsize=FONT_TITLE, fontweight='bold', pad=8)
    ax.grid(axis='x', color=COL_GRID, linewidth=0.7, zorder=0)
    ax.set_axisbelow(True)
    ax.spines[['top', 'right']].set_visible(False)
    ax.set_xlim(0, vals.max() * 1.18)
    ax.legend(handles=[
        mpatches.Patch(facecolor=COL_DYNAMIC,
                       label='Dynamic (meteorological)'),
        mpatches.Patch(facecolor=COL_STATIC,
                       label='Static (terrain/socioeconomic)')],
        fontsize=8, loc='lower right',
        framealpha=0.9, edgecolor='#cccccc')

def plot_pi(ax, pi_result, title):
    """Permutation importance bar chart."""
    means  = pi_result.importances_mean
    stds   = pi_result.importances_std
    idx    = np.argsort(means)
    labs   = [FEATURE_LABELS[FEATURES[i]] for i in idx]
    colors = [feature_colour(FEATURES[i]) for i in idx]
    ax.barh(range(len(idx)), means[idx], xerr=stds[idx],
            color=colors, edgecolor='white', linewidth=0.5,
            height=0.7, zorder=3,
            error_kw={'elinewidth':1.2, 'capsize':3,
                      'ecolor':'#666666', 'capthick':1.2})
    ax.axvline(0, color='#333333', linewidth=1.0,
               linestyle='--', zorder=4)
    for i, (m, s) in enumerate(zip(means[idx], stds[idx])):
        if m > 0:
            ax.text(m + s + 0.0003, i, f'{m:.4f}',
                    va='center', ha='left',
                    fontsize=7.5, color='#333333')
    ax.set_yticks(range(len(labs)))
    ax.set_yticklabels(labs, fontsize=FONT_TICK)
    ax.set_xlabel('Decrease in ROC-AUC when permuted\n'
                  '(mean ± std, 10 repeats)', fontsize=FONT_LABEL)
    ax.set_title(title, fontsize=FONT_TITLE, fontweight='bold', pad=8)
    ax.grid(axis='x', color=COL_GRID, linewidth=0.7, zorder=0)
    ax.set_axisbelow(True)
    ax.spines[['top', 'right']].set_visible(False)
    ax.legend(handles=[
        mpatches.Patch(facecolor=COL_DYNAMIC,
                       label='Dynamic (meteorological)'),
        mpatches.Patch(facecolor=COL_STATIC,
                       label='Static (terrain/socioeconomic)')],
        fontsize=8, loc='lower right',
        framealpha=0.9, edgecolor='#cccccc')

# ── Figure 1: Native importance — all three XGBoost models ────────────────────
print("\n  Figure 1: Native importance (all 3 XGBoost models)...")

fig1, axes1 = plt.subplots(1, 3, figsize=(24, 7))
fig1.subplots_adjust(wspace=0.38)

plot_xgb_native_importance(
    axes1[0], model_xgb,
    '(a) XGBoost Ignition Model\nNative Feature Importance (gain)')
for ax, key, letter in [(axes1[1],'human','b'), (axes1[2],'natural','c')]:
    info = cause_models[key]
    plot_xgb_native_importance(
        ax, info['model'],
        f'({letter}) XGBoost Cause: {info["pos_name"]} vs Other\n'
        f'Native Feature Importance (gain)')

fig1.suptitle(
    'XGBoost — Native Feature Importance (gain)\n'
    'Ignition Model + Binary Cause Models',
    fontsize=13, fontweight='bold', y=1.01)
fig1.text(0.5, -0.01, SOURCE_CAUSE,
          ha='center', fontsize=7.5, color='#888888', style='italic')
fig1.savefig('figures/fig_xgb_native_importance_all.png',
             dpi=300, bbox_inches='tight', facecolor='white')
print("    Saved: figures/fig_xgb_native_importance_all.png")
plt.close(fig1)

# =============================================================================
# STEP 8: Permutation importance
# =============================================================================
print("\n" + "="*60)
print("STEP 8: Permutation importance")
print("="*60)

# ── XGBoost ignition PI ────────────────────────────────────────────────────────
print("\n  Computing XGBoost ignition permutation importance...")
PERM_NOFIRE_SAMPLE = 50_000

fire_val_pi   = df_val[df_val['target_ignition'] == 1]
nofire_val_pi = df_val[df_val['target_ignition'] == 0].sample(
    n=min(PERM_NOFIRE_SAMPLE, (df_val['target_ignition']==0).sum()),
    random_state=RANDOM_STATE)
perm_df    = pd.concat([fire_val_pi, nofire_val_pi], ignore_index=True)
X_perm_ign = perm_df[FEATURES].values
y_perm_ign = perm_df['target_ignition'].values

print(f"  PI sample: {len(fire_val_pi):,} fire + "
      f"{len(nofire_val_pi):,} no-fire")
t0 = time.time()
pi_ign = permutation_importance(
    model_xgb, X_perm_ign, y_perm_ign,
    n_repeats=10, scoring='roc_auc',
    random_state=RANDOM_STATE, n_jobs=-1)
print(f"  Computed in {(time.time()-t0)/60:.1f} minutes")

# ── Cause model PI ─────────────────────────────────────────────────────────────
print("\n  Computing cause model permutation importance...")
pi_cause = {}
for key, info in cause_models.items():
    print(f"    {info['pos_name']} vs Other...")
    y_bin = (y_cause_val_a == info['pos_label']).astype(int)
    t0 = time.time()
    pi = permutation_importance(
        info['model'], X_cause_val_a, y_bin,
        n_repeats=10, scoring='roc_auc',
        random_state=RANDOM_STATE, n_jobs=-1)
    print(f"    Done in {(time.time()-t0)/60:.1f} minutes")
    pi_cause[key] = pi

# ── Figure 2: PI — all three models ───────────────────────────────────────────
print("\n  Figure 2: Permutation importance (all 3 models)...")

fig2, axes2 = plt.subplots(1, 3, figsize=(24, 7))
fig2.subplots_adjust(wspace=0.38)

plot_pi(axes2[0], pi_ign,
        '(a) XGBoost Ignition Model\nPermutation Importance')
for ax, key, letter in [(axes2[1],'human','b'), (axes2[2],'natural','c')]:
    info = cause_models[key]
    plot_pi(ax, pi_cause[key],
            f'({letter}) XGBoost Cause: {info["pos_name"]} vs Other\n'
            f'Permutation Importance')

fig2.suptitle(
    'XGBoost — Permutation Feature Importance (ROC-AUC)\n'
    'Ignition Model + Binary Cause Models  |  Validation: 2021–2022',
    fontsize=13, fontweight='bold', y=1.01)
fig2.text(0.5, -0.01, SOURCE_CAUSE,
          ha='center', fontsize=7.5, color='#888888', style='italic')
fig2.savefig('figures/fig_xgb_permutation_importance_all.png',
             dpi=300, bbox_inches='tight', facecolor='white')
print("    Saved: figures/fig_xgb_permutation_importance_all.png")
plt.close(fig2)

# ── Figure 3: Cause models — native + PI side by side (2×2) ───────────────────
print("\n  Figure 3: Cause importance comparison (2×2)...")

fig3, axes3 = plt.subplots(2, 2, figsize=(20, 14))
fig3.subplots_adjust(wspace=0.35, hspace=0.40)

for col, key, label in [(0,'human','Human vs Other'),
                         (1,'natural','Natural vs Other')]:
    plot_xgb_native_importance(
        axes3[0][col], cause_models[key]['model'],
        f'XGBoost Cause: {label}\nNative Importance (gain)')
    plot_pi(
        axes3[1][col], pi_cause[key],
        f'XGBoost Cause: {label}\nPermutation Importance (ROC-AUC)')

fig3.suptitle(
    'XGBoost Binary Cause Models — Feature Importance\n'
    'Native (gain) vs Permutation (ROC-AUC)  |  Validation: 2021–2022',
    fontsize=13, fontweight='bold', y=1.01)
fig3.text(0.5, -0.01, SOURCE_CAUSE,
          ha='center', fontsize=7.5, color='#888888', style='italic')
fig3.savefig('figures/fig_xgb_cause_importance_comparison.png',
             dpi=300, bbox_inches='tight', facecolor='white')
print("    Saved: figures/fig_xgb_cause_importance_comparison.png")
plt.close(fig3)

# =============================================================================
# STEP 9: Final summary
# =============================================================================
print(f"\n{'='*65}")
print("LaTeX — Binary Cause Confusion Matrices")
print(f"{'='*65}")
for model_key, e in binary_eval.items():
    pos = e['pos_name']
    print(f"\n  % {pos} vs Other  "
          f"(bal acc={e['bal_acc']:.4f}  AUC={e['val_auc']:.4f})")
    print(f"  \\begin{{tabular}}{{lrr}}")
    print(f"  \\toprule")
    print(f"  & \\textbf{{Pred Other}} & \\textbf{{Pred {pos}}} \\\\")
    print(f"  \\midrule")
    print(f"  \\textbf{{Actual Other}}   & {e['tn']:,} & {e['fp']:,} \\\\")
    print(f"  \\textbf{{Actual {pos}}} & {e['fn']:,} & {e['tp']:,} \\\\")
    print(f"  \\bottomrule")
    print(f"  \\end{{tabular}}")

print(f"\n{'='*65}")
print("FINAL SUMMARY")
print(f"{'='*65}")
print(f"\n  IGNITION MODEL (XGBoost)")
print(f"    Source      : model_ignition_final.pkl")
print(f"    CV threshold: {CV_THRESHOLD:.4f}")
print(f"    ROC-AUC     : {roc_auc_score(y_ign_val, y_prob_ign):.4f}")
print(f"    Recall      : {tp_i/(tp_i+fn_i):.4f}  "
      f"({tp_i}/{tp_i+fn_i} fires)")
print(f"    Precision   : "
      f"{tp_i/(tp_i+fp_i) if (tp_i+fp_i)>0 else 0:.4f}")
print(f"    FPR         : {fp_i/(fp_i+tn_i):.4f}")

print(f"\n  CAUSE MODELS (XGBoost binary, three-way decision)")
print(f"    Uncertainty threshold : {UNCERTAIN_THRESHOLD}")
for key, e in binary_eval.items():
    print(f"    {e['pos_name']:<25}: "
          f"AUC={e['val_auc']:.4f}  "
          f"Bal.Acc={e['bal_acc']:.4f}  "
          f"F1={e['f1']:.4f}")

print(f"\n  THREE-WAY ACCURACY")
print(f"    Overall accuracy      : {overall_acc:.4f}")
print(f"    Classified-only acc   : {known_acc:.4f}")
print(f"    Balanced accuracy     : {bal_acc_3way:.4f}")
print(f"    Uncertain rate        : "
      f"{n_uncertain/n_total*100:.1f}%  ({n_uncertain:,}/{n_total:,})")

print(f"\n  Models saved:")
print(f"    models/model_xgb_cause_binary_human.pkl")
print(f"    models/model_xgb_cause_binary_natural.pkl")

print(f"\n  Figures saved:")
for fig_name in [
    'fig_xgb_native_importance_all.png      — all 3 models native gain',
    'fig_xgb_permutation_importance_all.png — all 3 models PI',
    'fig_xgb_cause_importance_comparison.png — cause native vs PI (2×2)',
]:
    print(f"    figures/{fig_name}")